# CASMI MIST → MS-GPT evaluation

Encode each labeled MS/MS spectrum with pretrained MIST, threshold its Morgan fingerprint probabilities, and generate candidate SMILES with MS-GPT using CASMI's provided molecular formula. The oracle baseline gives the same decoder the reference Morgan fingerprint and the same formula and sampling budget.

**Choose how much data to evaluate** by setting `MODE = "full"` and editing `DATA_PERCENT` below, then choosing **Run All**. For example, `0.1` evaluates the first 0.1% of labeled spectra and `100` evaluates the entire file. Fractional row counts round up. Smoke mode evaluates up to 16 rows with one decoding attempt per arm. Each spectrum counts once, so molecules with more spectra have more weight. The competition test split has no reference labels for scoring.

The default fingerprint threshold, 0.20, comes from the repository's earlier small CASMI validation sweep. The full-file scores are descriptive because that calibration subset is included and overlap with model pretraining has not been checked.

## Environment

Use the repository's `.venv` kernel and open this notebook from the project root. To launch Jupyter from PowerShell:

```powershell
& .\.venv\Scripts\python.exe -m jupyterlab
```

If the environment has CPU-only PyTorch, install the [PyTorch 2.10 CUDA 12.8 build](https://pytorch.org/get-started/previous-versions/) and restart the kernel:

```powershell
& .\.venv\Scripts\python.exe -m pip install torch==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128
```

On Windows, install the [Triton 3.6 build for PyTorch 2.10](https://github.com/triton-lang/triton-windows) for Inductor compilation.

```powershell
& .\.venv\Scripts\python.exe -m pip install 'triton-windows>=3.6,<3.7'
```

The notebook uses the existing [MIST and CASMI assets](spectrum_encoder_module/README.md) and [MS-GPT assets](SMILES_decoder_module/README.md). Both networks run on CUDA. PyTorch compiles MIST inference and MS-GPT's per-token cross-attention blocks on first use. Parquet reads, spectrum preprocessing, and candidate scoring run on the CPU.

In [1]:
from pathlib import Path

MODE = "full"  # Use "smoke" for a 16-row check.
DATA_PERCENT = 0.1  # Percentage of labeled rows to evaluate in full mode.
THRESHOLD = 0.20
FULL_SAMPLES = 4  # Per spectrum, separately for the predicted and oracle fingerprints.
FULL_BATCH_SIZE = 16
COMPILE_BACKEND = "inductor"
CACHED_GENERATION = True  # Reuse MS-GPT conditioning projections across generated tokens.
SEED = 42

ROOT = Path.cwd().resolve()
ARTIFACTS = ROOT / "spectrum_encoder_module" / "artifacts"
DATA = ARTIFACTS / "train.parquet"
MIST_CHECKPOINT = ROOT / "mist_msg_noaug.ckpt"

if MODE not in {"smoke", "full"}:
    raise ValueError('Set MODE to "smoke" or "full".')

if not 0 < DATA_PERCENT <= 100:
    raise ValueError("DATA_PERCENT must be greater than 0 and at most 100.")

SAMPLES = 1 if MODE == "smoke" else FULL_SAMPLES
BATCH_SIZE = 8 if MODE == "smoke" else FULL_BATCH_SIZE

In [2]:
import os
from math import ceil
import sys

cache = ARTIFACTS / "compile_cache"
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", str(cache / "inductor"))
os.environ.setdefault("TRITON_CACHE_DIR", str(cache / "triton"))

import pyarrow.parquet as pq
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        f"CUDA is unavailable in {sys.executable} (PyTorch {torch.__version__}). "
        "Select the CUDA-enabled project kernel and restart it."
    )

if not MIST_CHECKPOINT.is_file():
    MIST_CHECKPOINT = ARTIFACTS / MIST_CHECKPOINT.name

decoder_dir = ROOT / "SMILES_decoder_module"
for asset in (
    DATA,
    MIST_CHECKPOINT,
    decoder_dir / "MS-GPT" / "ms_gpt_mlm.pt",
    decoder_dir / "MS-GPT" / "mlm_pretraining.yaml",
    decoder_dir / ".tokenizer" / "tokenizer.json",
    decoder_dir / ".upstream" / "src" / "ms_gpt",
):
    if not asset.exists():
        raise FileNotFoundError(f"Missing {asset}. See the setup links above.")

with pq.ParquetFile(DATA) as source:
    source_rows = source.metadata.num_rows

LIMIT = None
subset_suffix = ""
if MODE == "smoke":
    LIMIT = 16
elif DATA_PERCENT < 100 and source_rows:
    LIMIT = ceil(source_rows * DATA_PERCENT / 100)
    subset_suffix = f"_first_{LIMIT}_rows"

generation_suffix = "_cached" if CACHED_GENERATION else ""
OUTPUT_DIR = ARTIFACTS / f"pipeline_{MODE}_{COMPILE_BACKEND}{generation_suffix}{subset_suffix}"
DATABASE = OUTPUT_DIR / "results.sqlite3"
planned_rows = source_rows if LIMIT is None else min(LIMIT, source_rows)

print(f"GPU: {torch.cuda.get_device_name()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
print(f"Mode: {MODE} | Rows: {planned_rows:,} of {source_rows:,} | Results: {DATABASE}")

GPU: NVIDIA GeForce RTX 4060 | PyTorch 2.10.0+cu128 | CUDA 12.8
Mode: full | Rows: 2,540 of 2,539,608 | Results: D:\Habib\7th Sem\AI\Project\AI-Project\spectrum_encoder_module\artifacts\pipeline_full_inductor_cached_first_2540_rows\results.sqlite3


## Evaluate

Each completed batch is committed to the results database. To resume after stopping, rerun with the same settings and output path. Percentage runs use a path based on their row limit. Use a new `OUTPUT_DIR` and `DATABASE` for a fresh evaluation or when changing the threshold, samples, batch size, or seed. Cached generation uses its own database because floating-point differences can change sampled candidates. Set `DATA_PERCENT = 100` and `CACHED_GENERATION = False` to resume an earlier full-file run made before cached generation was added. Full runtime depends on spectrum preprocessing and SAFE generation length. The progress lines report observed throughput, including any compilation during processing.

In [3]:
from spectrum_encoder_module.full_pipeline import run_pipeline

summary = run_pipeline(
    DATA,
    MIST_CHECKPOINT,
    DATABASE,
    max_rows=LIMIT,
    threshold=THRESHOLD,
    samples=SAMPLES,
    batch_size=BATCH_SIZE,
    seed=SEED,
    compile_backend=COMPILE_BACKEND,
    cached_generation=CACHED_GENERATION,
)

NVIDIA GeForce RTX 4060 | 0/2,540 spectra already processed
🚨 `inputs` is part of SAFEDoubleHeadsModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in d:\Habib\7th Sem\AI\Project\AI-Project\.venv\Lib\site-packages\safe\trainer\model.py.


D:\Habib\7th Sem\AI\Project\AI-Project\SMILES_decoder_module\.upstream\src\ms_gpt\models\conditional_mlm.py:75: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.self_attn = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Compiling MIST and MS-GPT cross-attention with inductor on first use.
16/2,540 | 16 decoded | 0.7 rows/s


W1112 22:26:51.913000 26932 Lib\site-packages\torch\_inductor\utils.py:1679] [10/1] Not enough SMs to use max_autotune_gemm mode
d:\Habib\7th Sem\AI\Project\AI-Project\.venv\Lib\site-packages\torch\_inductor\lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


256/2,540 | 248 decoded | 1.8 rows/s
512/2,540 | 497 decoded | 1.9 rows/s
768/2,540 | 747 decoded | 2.1 rows/s
2,540/2,540 | 2,470 decoded | 2.4 rows/s


## Results

Fingerprint statistics average over encoded spectra. Decoder rates use spectra eligible for both models, except **Recovery / all rows**, which also counts skipped spectra in the denominator. Recovery checks 14-character InChIKey connectivity among all valid candidates from the configured sampling budget. Top-1 uses the decoder's ranking by formula match and sample frequency.

In [4]:
import json

from IPython.display import Markdown, display

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

scanned = summary["rows_scanned"]
print(f"Scanned: {scanned:,} | Encoded: {summary['encoded']:,} | Evaluated: {summary['evaluated']:,}")
print(f"Row status counts: {summary['status_counts']}")
if summary["fingerprint"]:
    print("Mean fingerprint metrics:", {name: round(value, 4) for name, value in summary["fingerprint"].items()})
if summary["rows_per_second"] is not None:
    print(f"Processing: {summary['elapsed_seconds']:,.1f} s at {summary['rows_per_second']:.2f} rows/s")
print(f"Peak PyTorch CUDA allocation: {summary['peak_cuda_allocated_mib']:,.0f} MiB")

metrics = {
    "Top-1 recovery": "top1_rate",
    f"Recovery in {SAMPLES} samples": "top_k_recovery_rate",
    "Formula match": "formula_match_rate",
    "Valid samples": "valid_sample_rate",
    "Recovery / all rows": "recovery_rate_all_rows",
}
table = ["| Metric | MIST → MS-GPT | Oracle → MS-GPT |", "| :-- | --: | --: |"]
for label, key in metrics.items():
    values = [summary["arms"][arm][key] for arm in ("predicted", "oracle")]
    rates = [f"{value:.2%}" if value is not None else "—" for value in values]
    table.append(f"| {label} | {rates[0]} | {rates[1]} |")
display(Markdown("\n".join(table)))
print(f"Summary: {summary_path}")

Scanned: 2,540 | Encoded: 2,470 | Evaluated: 2,470
Row status counts: {'evaluated': 2470, 'no_assigned_fragments': 31, 'unsupported_formula': 31, 'unsupported_adduct': 8}
Mean fingerprint metrics: {'tanimoto': 0.1888, 'precision': 0.367, 'recall': 0.3003, 'predicted_bits': 32.0, 'true_bits': 42.104}
Processing: 1,048.4 s at 2.42 rows/s
Peak PyTorch CUDA allocation: 2,777 MiB


| Metric | MIST → MS-GPT | Oracle → MS-GPT |
| :-- | --: | --: |
| Top-1 recovery | 0.00% | 85.95% |
| Recovery in 4 samples | 0.00% | 87.41% |
| Formula match | 9.72% | 97.41% |
| Valid samples | 64.34% | 99.16% |
| Recovery / all rows | 0.00% | 85.00% |

Summary: D:\Habib\7th Sem\AI\Project\AI-Project\spectrum_encoder_module\artifacts\pipeline_full_inductor_cached_first_2540_rows\summary.json


In [5]:
from contextlib import closing
import sqlite3

# Read one scored row to inspect the ranked SMILES from both arms.
with closing(sqlite3.connect(DATABASE)) as connection:
    example = connection.execute(
        "SELECT row_index, result FROM spectra WHERE status = ? ORDER BY row_index LIMIT 1",
        ("evaluated",),
    ).fetchone()

if example is not None:
    display({"parquet_row": example[0], **json.loads(example[1])})

{'parquet_row': 0,
 'inchikey14': 'AAALVYBICLMAMA',
 'provided_formula': 'C20H15N3O2',
 'fingerprint': {'tanimoto': 0.1875,
  'precision': 0.28125,
  'recall': 0.36,
  'predicted_bits': 32.0,
  'true_bits': 25.0},
 'status': 'evaluated',
 'predicted': {'top1': False,
  'recovered': False,
  'has_formula_match': False,
  'valid_samples': 0,
  'candidates': []},
 'oracle': {'top1': True,
  'recovered': True,
  'has_formula_match': True,
  'valid_samples': 4,
  'candidates': [{'key': 'AAALVYBICLMAMA',
    'formula': 'C20H15N3O2',
    'smiles': 'O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21',
    'count': 4}]}}